# Analyst-Ready SEC Filings RAG System

**Course:** AI Engineering  
**Professor:** Apostolos Filippas  
**Institution:** Fordham University  
**Team:** Joel Ebukatobi, Claudia Gisemba

This notebook builds a complete SEC 10-K Retrieval-Augmented Generation pipeline for analyst workflows.

In [20]:
%load_ext autoreload
%autoreload 2

In [59]:
# Section 1: Setup & Configuration
import os
import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv

from src.ingest import load_raw_filings, filing_stats, load_live_filings
from src.chunk import parse_sections, section_coverage_stats, build_chunks, chunk_stats
from src.embed import embed_chunks, embedding_info
from src.retrieve import HybridRetriever
from src.generate import compare, generate_structured_output
from src.evaluate import build_test_set, run_retrieval_eval, ablation_table

load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    print("Warning: OPENAI_API_KEY is not set. Generation sections will fail until set.")
else:
    print("OpenAI API key loaded successfully.")

# Global constants for a reproducible project configuration.
CHUNK_SIZE = 800
CHUNK_OVERLAP = 200
TOP_K = 10
EMBED_MODEL = "all-MiniLM-L6-v2"
LLM_MODEL = "gpt-4o-mini"
# TICKERS = ["TSLA", "AAPL", "MSFT", "AMZN", "NVDA"]
# YEAR_RANGE = (2018, 2023)
# TARGET_SECTIONS = ["section_1a", "section_7", "section_3"]

Path("data").mkdir(parents=True, exist_ok=True)
print("Constants defined and environment loaded.")

OpenAI API key loaded successfully.
Constants defined and environment loaded.


In [53]:
TICKERS = ["TSLA", "AAPL", "AMZN", "NVDA", "CCL"]
YEAR_RANGE = (2018, 2025)
TARGET_SECTIONS = ["section_1","section_1a", "section_7", "section_3","section_8"]

## Section 2: Data Ingestion
Load 10-K filings from Hugging Face and normalize into a DataFrame with core metadata.

In [22]:
from src.ingest import load_live_filings, filing_stats

In [23]:
import inspect
print(inspect.getsource(load_live_filings))

def load_live_filings(tickers: Iterable[str], year_range: Tuple[int, int]) -> pd.DataFrame:
    """
    Enhanced Ingestion: Captures structural sections and metadata 
    to support the Credit Signal Taxonomy.
    """
    rows = []
    year_min, year_max = year_range

    for ticker in tqdm(tickers, desc="Ingesting SEC Corpus"):
        try:
            company = Company(ticker)
            filings = company.get_filings(form="10-K")
            if not filings:
                continue

            for filing in filings:
                f_year = filing.filing_date.year
                if year_min <= f_year <= year_max:
                    
                    # Get the structured TenK object
                    tenk = filing.obj()
                    
                    # --- SIGNAL TAXONOMY MAPPING ---
                    # Item 1: Business (Business Fragility Signals)
                    item_1 = tenk["Item 1"] or ""
                    
                    # Item 1A: Risk Factors (R

In [ ]:
# NOTE: An earlier version used load_raw_filings() from the HuggingFace edgar-corpus dataset.
# That approach was replaced by load_live_filings() which pulls directly from SEC EDGAR
# via the edgartools library, giving us structured section access (Items 1A, 3, 7, 8).

In [26]:
df_test = load_live_filings(tickers=["AAPL"], year_range=(2023, 2024))
print(df_test.columns)

Ingesting SEC Corpus: 100%|██████████| 1/1 [00:22<00:00, 22.65s/it]

Index(['ticker', 'year', 'company', 'section_1', 'section_1a', 'section_3',
       'section_7', 'section_8', 'len_1a', 'len_7', 'len_3'],
      dtype='str')


In [27]:
df_filings = load_live_filings(
    tickers=TICKERS,
    year_range=YEAR_RANGE
)

print("Raw filing DataFrame shape:", df_filings.shape)
display(df_filings.head(3))
display(filing_stats(df_filings).head(20))

Ingesting SEC Corpus:   0%|          | 0/5 [00:00<?, ?it/s]TenK falling back to legacy parser for 'Item 1' (filing: 0001104659-25-042659). New parser sections available: ['part_iii_part_iii', 'part_iii_item_10', 'part_iii_item_11', 'part_iii_item_12', 'part_iii_item_13', 'part_iii_item_14', 'part_iv_part_iv', 'part_iv_item_15']. This fallback will be removed in v6.0.
TenK falling back to legacy parser for 'Item 1A' (filing: 0001104659-25-042659). New parser sections available: ['part_iii_part_iii', 'part_iii_item_10', 'part_iii_item_11', 'part_iii_item_12', 'part_iii_item_13', 'part_iii_item_14', 'part_iv_part_iv', 'part_iv_item_15']. This fallback will be removed in v6.0.
TenK falling back to legacy parser for 'Item 1A' (filing: 0001104659-25-042659). New parser sections available: ['part_iii_part_iii', 'part_iii_item_10', 'part_iii_item_11', 'part_iii_item_12', 'part_iii_item_13', 'part_iii_item_14', 'part_iv_part_iv', 'part_iv_item_15']. This fallback will be removed in v6.0.
TenK f

Raw filing DataFrame shape: (40, 11)


,ticker,year,company,section_1,section_1a,section_3,section_7,section_8,len_1a,len_7,len_3
0,AAPL,2018,Apple Inc.,Item 1.\n\nBusiness\n\nCompany Background\n\nT...,Item 1A.\n\nRisk Factors\n\nThe following disc...,Item 3.\n\nLegal Proceedings\n\nThe Company is...,Item 7.\n\nManagement’s Discussion and Analysi...,Item 8.\n\nFinancial Statements and Supplement...,55472,58061,1359
1,AAPL,2019,Apple Inc.,Item 1.\n\nBusiness\n\nCompany Background\n\nT...,Item 1A.\n\nRisk Factors\n\nThe following disc...,Item 3.\n\nLegal Proceedings\n\nThe Company is...,Item 7.\n\nManagement’s Discussion and Analysi...,Item 8.\n\nFinancial Statements and Supplement...,54994,34310,1348
2,AAPL,2020,Apple Inc.,Item 1. Business\n\nCompany Background\n\nT...,Item 1A. Risk Factors\n\nThe following disc...,Item 3. Legal Proceedings\n\nThe Company is...,Item 7. Management’s Discussion and Analysi...,Item 8. Financial Statements and Supplement...,60994,31977,898


,ticker,year,count
0,AAPL,2018,1
1,AAPL,2019,1
2,AAPL,2020,1
3,AAPL,2021,1
4,AAPL,2022,1
5,AAPL,2023,1
6,AAPL,2024,1
7,AAPL,2025,1
8,AMZN,2018,1
9,AMZN,2019,1


## Section 3: Section Parsing
Extract Item 1A (Risk Factors), Item 7 (MD&A), and Item 3 (Legal Proceedings) from filing text.

In [37]:
df_sections = parse_sections(df_filings, target_sections=TARGET_SECTIONS)
print("Parsed sections shape:", df_sections.shape)
display(df_sections.head(5))
display(section_coverage_stats(df_sections))

Parsing sections: 100%|██████████| 40/40 [00:01<00:00, 32.50it/s]

Parsed sections shape: (200, 5)


,ticker,year,section_type,section_text,section_char_len
0,AAPL,2018,section_1,Item 1. Business Company Background The Compan...,30185
1,AAPL,2018,section_1a,Item 1A. Risk Factors The following discussion...,55472
2,AAPL,2018,section_7,Item 7. Management’s Discussion and Analysis o...,58061
3,AAPL,2018,section_3,Item 3. Legal Proceedings The Company is subje...,1359
4,AAPL,2018,section_8,Item 8. Financial Statements and Supplementary...,99155


,section_type,coverage
0,section_1,40
1,section_1a,40
2,section_3,40
3,section_7,40
4,section_8,40


## Section 4: Section-Aware Chunking
Chunk each parsed section independently so chunks never cross section boundaries.

In [38]:
# Define path
cache_file = "data/chunks.pkl"

# Logic: If I changed my taxonomy, I might want to force a refresh
force_refresh = False 

if force_refresh and os.path.exists(cache_file):
    os.remove(cache_file)
    print("Old cache removed to apply new Credit Taxonomy.")

all_chunks = build_chunks(
    df_sections,
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    cache_path=cache_file,
    use_multiprocessing=True
)

print("Total chunk count:", len(all_chunks))
display(pd.DataFrame(all_chunks).head(3))
display(chunk_stats(all_chunks))

Chunking sections: 100%|██████████| 200/200 [00:09<00:00, 21.62it/s]


Total chunk count: 2363


,ticker,year,section_type,chunk_id,text,meta_section_len,label
0,AAPL,2018,section_1,AAPL-2018-section_1-0,Item 1. Business Company Background The Compan...,30185,Business Description
1,AAPL,2018,section_1,AAPL-2018-section_1-1,"system, which includes iPad Pro®, iPad and iPa...",30185,Business Description
2,AAPL,2018,section_1,AAPL-2018-section_1-2,"iOS devices and Mac computers, features e-book...",30185,Business Description


,metric,value
0,count,2363.00
1,min_tokens,19.00
2,mean_tokens,765.05
3,max_tokens,800.00


In [39]:
# Convert to DataFrame to verify the new 'len' columns are there
df_chunk_verify = pd.DataFrame(all_chunks)
print("Columns in chunks:", df_chunk_verify.columns.tolist())

Columns in chunks: ['ticker', 'year', 'section_type', 'chunk_id', 'text', 'meta_section_len', 'label']


## Section 5: Embedding
Generate semantic vectors for every chunk and cache to disk.

In [40]:
vectors, vector_metadata = embed_chunks(
    all_chunks,
    model_name=EMBED_MODEL,
    batch_size=256,
    cache_path="data/embeddings.pkl"
)

print("Vectors shape:", getattr(vectors, "shape", None))
print("Embedding info:", embedding_info(vectors))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2969.61it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 10/10 [04:37<00:00, 27.76s/it]

Vectors shape: (2363, 384)
Embedding info: {'count': 2363, 'dimension': 384, 'workers': 8}


## Section 6: Hybrid Retrieval
Combine FAISS semantic retrieval with BM25 lexical retrieval using weighted reranking.

In [41]:
retriever = HybridRetriever(
    vectors=vectors,
    metadata=vector_metadata,
    embed_model_name=EMBED_MODEL,
    semantic_weight=0.6,
    lexical_weight=0.4
)

sample_queries = [
    "major macroeconomic risk factors",
    "changes in segment revenue drivers",
    "notable legal proceedings and litigation"
]

for q in sample_queries:
    results = retriever.retrieve(query=q, ticker="TSLA", year=2022, section_type="section_1a", top_k=TOP_K)
    print("\nQuery:", q)
    display(pd.DataFrame(results)[["chunk_id", "semantic_score", "bm25_score", "final_score"]].head(TOP_K))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2786.14it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Query: major macroeconomic risk factors


,chunk_id,semantic_score,bm25_score,final_score
0,TSLA-2022-section_1a-0,0.495252,1.000000,0.697151
1,TSLA-2022-section_1a-16,0.095460,0.601519,0.297883
2,TSLA-2022-section_1a-19,0.191694,0.447224,0.293906
3,TSLA-2022-section_1a-6,0.341360,0.154295,0.266534
4,TSLA-2022-section_1a-15,0.303501,0.191353,0.258642
5,TSLA-2022-section_1a-12,0.189795,0.257251,0.216778
6,TSLA-2022-section_1a-11,0.186115,0.257530,0.214681
7,TSLA-2022-section_1a-21,0.329340,0.037738,0.212699
8,TSLA-2022-section_1a-14,0.304083,0.037058,0.197273
9,TSLA-2022-section_1a-8,0.302498,0.037058,0.196322



Query: changes in segment revenue drivers


,chunk_id,semantic_score,bm25_score,final_score
0,TSLA-2022-section_1a-18,0.250871,1.000000,0.550522
1,TSLA-2022-section_1a-5,0.237557,0.658352,0.405875
2,TSLA-2022-section_1a-21,0.370442,0.000889,0.222621
3,TSLA-2022-section_1a-17,0.306594,0.085026,0.217967
4,TSLA-2022-section_1a-8,0.325019,0.052739,0.216107
5,TSLA-2022-section_1a-14,0.294558,0.057431,0.199707
6,TSLA-2022-section_1a-2,0.276392,0.046807,0.184558
7,TSLA-2022-section_1a-12,0.244391,0.063051,0.171855
8,TSLA-2022-section_1a-10,0.285853,0.000000,0.171512
9,TSLA-2022-section_1a-15,0.276199,0.010216,0.169806



Query: notable legal proceedings and litigation


,chunk_id,semantic_score,bm25_score,final_score
0,TSLA-2022-section_1a-19,0.162971,1.000000,0.497783
1,TSLA-2022-section_1a-16,0.233935,0.624004,0.389962
2,TSLA-2022-section_1a-18,0.185492,0.672231,0.380187
3,TSLA-2022-section_1a-15,0.147765,0.617939,0.335835
4,TSLA-2022-section_1a-20,0.183737,0.510054,0.314264
5,TSLA-2022-section_1a-17,0.034612,0.626040,0.271183
6,TSLA-2022-section_1a-12,0.262390,0.006424,0.160004
7,TSLA-2022-section_1a-10,0.153538,0.119166,0.139789
8,TSLA-2022-section_1a-11,0.103814,0.115235,0.108382
9,TSLA-2022-section_1a-5,0.082429,0.121907,0.098220


In [42]:
# Example Query updated for Credit Analysis
query = "Identify changes in debt maturity, liquidity risks, and management tone regarding credit facilities."
results = retriever.retrieve(query, ticker="CCL", top_k=TOP_K)

In [43]:
results

[{'ticker': 'CCL',
  'year': 2025,
  'section_type': 'section_7',
  'chunk_id': 'CCL-2025-section_7-9',
  'source': 'CCL_2025_10k',
  'text': 'as well as estimated interest payments and does not include the impact of any future possible refinancings. Excludes undrawn export credits. (b)As of November 30, 2024, we have undrawn export credit facilities of $7.8 billion which fund a portion of our newbuild contractual commitments. Funding Sources As of November 30, 2024, we had $4.2 billion of liquidity including $1.2 billion of cash and cash equivalents and $2.9 billion of borrowings available under our Revolving Facility. In addition, we had $7.8 billion of undrawn export credit facilities to fund ship deliveries planned through 2033. We plan to use existing liquidity and future cash flows from operations to fund our cash requirements including capital expenditures not funded by our export credit facilities. We seek to manage our credit risk exposures, including counterparty nonperforman

## Section 7: Comparative Reasoning
Retrieve evidence from two filing years and ask GPT-4o for structured differences.

In [60]:
comparison_result = compare(
    retriever=retriever,
    ticker="CCL",
    section_type="section_1a",
    year_a=2020,
    year_b=2025,
    query="material risk changes over time",
    top_k=TOP_K,
    model=LLM_MODEL
)

print("Raw diff output:")
print(comparison_result.get("raw_diff", "")[:3000])

Raw diff output:
### Delta Analysis of SEC Filings for CCL (2020 vs 2025)

#### 1. BUSINESS FRAGILITY
- **Customer Concentration**: No significant changes detected in customer concentration from 2020 to 2025.
- **Supply Chain Shifts**: 
  - **2020**: Mentioned potential delays or cancellations of cruises due to supply chain issues (chunk_id=CCL-2020-section_1a-2).
  - **2025**: Expanded on supply chain issues, emphasizing reliance on suppliers and potential disruptions (chunk_id=CCL-2025-section_1a-6). This indicates an increased fragility in the supply chain.
- **Competitive Pressure**: No explicit mention of competitive pressure in either year.

#### 2. RISK INTENSITY
- **New Risk Factors**: 
  - **2025** introduces risks related to cybersecurity and data privacy breaches (chunk_id=CCL-2025-section_1a-3), which were not present in 2020.
- **Expanded Risk Language**: 
  - **2025** includes more detailed language regarding climate change and regulatory impacts (chunk_id=CCL-2025-sectio

## Section 8: Structured Output Generation
Convert reasoning output into strict JSON schema and validate it with graceful error handling.

In [61]:
structured_output = generate_structured_output(comparison_result, model=LLM_MODEL)

if "error" in structured_output:
    print("Structured output failed gracefully:")
    print(json.dumps(structured_output, indent=2))
else:
    print("Validated structured JSON output:")
    print(json.dumps(structured_output, indent=2)[:5000])

Validated structured JSON output:
{
  "executive_summary": "CCL's risk landscape has evolved from 2020 to 2025, with increased supply chain fragility, new cybersecurity risks, and indications of potential liquidity stress, leading to a cautious outlook.",
  "findings": [
    {
      "category": "STRATEGIC",
      "materiality": "MEDIUM",
      "title": "Increased Supply Chain Risks",
      "evidence": "Expanded on supply chain issues, emphasizing reliance on suppliers and potential disruptions.",
      "verdict": "Classified as MEDIUM MATERIALITY due to strategic shifts in supply chain management.",
      "source": "CCL-2025-section_1a-6"
    },
    {
      "category": "REGULATORY",
      "materiality": "MEDIUM",
      "title": "New Cybersecurity Risks",
      "evidence": "Introduces risks related to cybersecurity and data privacy breaches.",
      "verdict": "Classified as MEDIUM MATERIALITY due to the introduction of new risk factors.",
      "source": "CCL-2025-section_1a-3"
    },


## Section 9: Evaluation
Evaluate retrieval and structured generation quality. Include ablation-style comparison table.

In [62]:
test_years = list(range(YEAR_RANGE[0], YEAR_RANGE[1] + 1))
test_set = build_test_set(tickers=TICKERS, years=test_years)
eval_results = run_retrieval_eval(retriever, test_set, top_k=TOP_K)

# Aggregate core retrieval metrics.
summary = pd.DataFrame({
    "Recall@5": [eval_results["Recall@5"].mean()],
    "MRR": [eval_results["MRR"].mean()]
})

print("Retrieval evaluation summary:")
display(summary)

# Real ablation: re-runs eval with BM25 and metadata filtering toggled off.
ablations = ablation_table(retriever, test_set, top_k=TOP_K)
display(ablations)

# Structured output and citation quality checks from one sample output.
from src.evaluate import citation_grounding_score, structured_output_accuracy

if isinstance(structured_output, dict) and "error" not in structured_output:
    cg = citation_grounding_score(structured_output)
    sa = structured_output_accuracy(structured_output)
else:
    cg, sa = 0.0, 0.0

print({
    "Citation Grounding Score": cg,
    "Structured Output Accuracy": sa
})

Running retrieval eval: 100%|██████████| 20/20 [00:02<00:00,  6.93it/s]


Retrieval evaluation summary:


,Recall@5,MRR
0,0.95,0.9


Semantic Only, No Filter: 100%|██████████| 20/20 [00:00<00:00, 24.93it/s]


,setup,Recall@5,MRR
0,Hybrid + Metadata Filter,0.95,0.90
1,No BM25 (Semantic Only),0.95,0.85
2,No Metadata Filtering,1.00,1.00
3,"Semantic Only, No Filter",1.00,1.00


{'Citation Grounding Score': 0.0, 'Structured Output Accuracy': 0.0}


## Section 10: Streamlit App
The full pipeline is wired in app.py for deployment on Streamlit Community Cloud.

In [ ]:
# Run this command in terminal (not inside notebook):
# streamlit run app.py

print("Streamlit app entrypoint: app.py")
print("UI includes ticker, years, section, spinner, structured output, and raw JSON toggle.")